In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
import sys


sys.path.append("C:/Users/tjsss/OneDrive/Desktop/machine learning/mlFromScratch")
from utils.dataLoader import data_loader


## random value generator

In [3]:
from re import X
import numpy as np

# random seed for reproducibility
np.random.seed(42)

# 1. Create a 20x5 feature matrix X with random numbers
x = np.random.randn(100, 5)

# 2. Define true feature weights to create a correlation
# Features 0, 1, and 2 are highly correlated to y, feature 3 is weak, feature 4 is irrelevant
true_weights = np.array([[2.5], [-1.8], [0.5], [0.1], [0.0]])

# 3. Create a 20x1 noise matrix
noise = np.random.randn(100, 1) * 0.35

# 4. Generate y with a linear correlation to X plus noise (y = Xw + noise)
z = np.dot(x, true_weights) + noise

y = 1 / (1 + np.exp(-z))

# Print shapes to verify dimensions
print(f"x shape: {x.shape}")
print(f"y shape: {y.shape}")


x shape: (100, 5)
y shape: (100, 1)


In [4]:
from utils.gradientDescent import SGD

In [ ]:


class LogisticRegressionFromScratch():
    
    def __init__ (self, num_feats ,lr = 0.001 ,sigma = 0.01):
        self.trained = False
        self.lr = lr
        self.num_feats = num_feats
        self.sigma = sigma
    
        if not self.trained:
            self.w = torch.normal(0, self.sigma, (self.num_feats , 1), requires_grad=True)
            self.b = torch.zeros(1, requires_grad=True)
        else:
            self.w = self.w_final
            self.b = self.b_final
        
    def calc(self, x):
        z = (torch.matmul(x, self.w) + self.b)
        y = ( 1 / (1 + torch.exp(-z)))
        return y
    
    def loss_fn(self,y_hat, y):
        l = -(y*torch.log(y_hat) + (1-y)*torch.log(1-y_hat))
        return l.mean()
    
    def config_optimizer(self,):
        return SGD([self.w, self.b], lr=self.lr)
        
    
    def train(self, train_data, val_data, epochs):
        optimizer = self.config_optimizer()
        batch = train_data[0]
        b_size = len(batch)
        
        loss_mean = 0.0
        self.loss_mean_list = []
        val_loss_mean = 0.0
        self.val_loss_mean_list = []
        
        num_bts = len(train_data)
        for epoch in range(epochs):
            for bts in range(num_bts):
                batch = train_data[bts]
                x = batch[0]
                y = batch[1]
                
                y_hat = self.calc(x)
                loss = self.loss_fn(y_hat, y)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                loss_mean += loss
            loss_mean = loss_mean / num_bts
            self.loss_mean_list.append(loss_mean)
            
            val_loss_mean = self.evaluate(val_data)
            self.val_loss_mean_list.append(val_loss_mean)
            
            print(f'Mean Train Loss in Epoch: {epoch + 1} is: {loss_mean}')
            print(f'Mean Val Loss in Epoch: {epoch + 1} is: {val_loss_mean}')      
            
        self.trained = True
        self.w_final = self.w
        self.b_final = self.b
        print('='*40)
        print('\n')
        print(f'final train loss: {loss_mean} \nfinal val loss: {val_loss_mean}')
        print(f'final weights: {self.w.detach().numpy()} \nfinal bias: {self.b.detach().numpy()}') 
        
    @torch.no_grad()
    def evaluate(self, data):
        batch = data[0]
        b_size = len(batch)
    
    
        val_loss_mean = 0
        total_loss = 0
        num_bts = len(data)
        for bts in range(num_bts):
            batch = data[bts]
            x = batch[0]
            y = batch[1]
    
            y_hat = self.calc(x)
            loss = self.loss_fn(y, y_hat)
    
            val_loss_mean += loss
    
        val_loss_mean = val_loss_mean / len(data)
        return val_loss_mean     
                
        
    @torch.no_grad()
    def predict(self, x):
        x = torch.tensor(x, dtype=torch.float32)
        y = self.calc(x)
        return y.detach().tolist()



    @torch.no_grad()
    def history(self,):
        epoch_train_loss = self.loss_mean_list
        epoch_val_loss = self.val_loss_mean_list
        num_epochs = len(epoch_train_loss)
        epochs = [i+1 for i in range(num_epochs)]

        plt.figure(figsize=(10,10))
        plt.plot(epochs, epoch_train_loss, '-', linewidth = 2, color = 'red', label = 'Train Loss')
        plt.plot(epochs, epoch_val_loss, '-', linewidth = 2, color = 'green', label = 'Val Loss')


        plt.grid(visible=True, axis='both', linestyle='-', color='gray', linewidth=0.6)
        min_loss = 0
        max_loss = 10
        plt.ylim(min_loss , max_loss)

        y_ticks = np.arange(min_loss, max_loss, 0.25)
        plt.yticks(y_ticks)
        x_ticks = np.arange(0, num_epochs+5, 5)
        plt.xticks(x_ticks)

        plt.title('Train and Val Loss')
        plt.legend()

        plt.show()
        
        

In [18]:
dl = data_loader(x, y, 0.8, 2)
train_dl = dl.train_loader()
test_dl = dl.test_loader()

In [19]:
lr = LogisticRegressionFromScratch(5, 0.01)

In [20]:
lr.train(train_dl,test_dl, 100)

Mean Train Loss in Epoch: 1 is: 0.6673358678817749
Mean Val Loss in Epoch: 1 is: 1.3334882259368896
Mean Train Loss in Epoch: 2 is: 0.6395300030708313
Mean Val Loss in Epoch: 2 is: 1.2533485889434814
Mean Train Loss in Epoch: 3 is: 0.6031332015991211
Mean Val Loss in Epoch: 3 is: 1.1827216148376465
Mean Train Loss in Epoch: 4 is: 0.5732613205909729
Mean Val Loss in Epoch: 4 is: 1.1204980611801147
Mean Train Loss in Epoch: 5 is: 0.5487290024757385
Mean Val Loss in Epoch: 5 is: 1.065544843673706
Mean Train Loss in Epoch: 6 is: 0.5283350348472595
Mean Val Loss in Epoch: 6 is: 1.0168230533599854
Mean Train Loss in Epoch: 7 is: 0.5111768841743469
Mean Val Loss in Epoch: 7 is: 0.9734283685684204
Mean Train Loss in Epoch: 8 is: 0.49657878279685974
Mean Val Loss in Epoch: 8 is: 0.9345919489860535
Mean Train Loss in Epoch: 9 is: 0.4840310215950012
Mean Val Loss in Epoch: 9 is: 0.8996695280075073
Mean Train Loss in Epoch: 10 is: 0.47314557433128357
Mean Val Loss in Epoch: 10 is: 0.86812198162078

In [21]:
lr.history()

AttributeError: 'LogisticRegressionFromScratch' object has no attribute 'loss_mean_store'